# Track4World — Colab 推理环境
**GPU**: T4 16GB (免费版) 或 A100 (Colab Pro)

**用途**: 在本机显存不足时，使用 Colab 跑大视频推理，结果下载回本地

## 使用流程
1. 运行 Cell 1：检查 GPU
2. 运行 Cell 2：克隆仓库 + 安装依赖（首次约15分钟）
3. 运行 Cell 3：下载预训练权重
4. 运行 Cell 4：上传视频文件
5. 运行 Cell 5：执行推理
6. 运行 Cell 6：下载结果

In [ ]:
# Cell 1: 检查 GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Cell 2: 克隆 Track4World + 安装依赖
import os

# 设置 HuggingFace 镜像（如超时可取消注释）
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

if not os.path.exists("/content/Track4World"):
    !git clone https://github.com/TencentARC/Track4World.git /content/Track4World
else:
    print("Track4World already cloned, skipping.")

os.chdir("/content/Track4World")
print("当前目录:", os.getcwd())

# 临时替换 requirements.txt 中锁死的 open3d 版本（0.18.0 不支持 Python 3.12）
!sed -i 's/open3d==0.18.0/open3d>=0.19.0/' requirements.txt
!grep open3d requirements.txt  # 确认替换成功

# 安装依赖
!pip install -q -r requirements.txt
!pip install -q viser tqdm matplotlib plotly
print("\n依赖安装完成")

In [ ]:
# Cell 3: 下载预训练权重
import os
os.makedirs("/content/Track4World/checkpoints", exist_ok=True)

ckpt_path = "/content/Track4World/checkpoints/track4world_moge.pth"
if not os.path.exists(ckpt_path):
    !wget -q --show-progress -O {ckpt_path} https://huggingface.co/TencentARC/Track4World/resolve/main/track4world_moge.pth
else:
    print("权重已存在，跳过下载")

print(f"权重大小: {os.path.getsize(ckpt_path)/1e6:.1f} MB")

In [ ]:
# Cell 4: 上传视频
from google.colab import files
import os

os.makedirs("/content/Track4World/input_videos", exist_ok=True)
print("请选择要上传的视频文件 (.mp4)")
uploaded = files.upload()

for fname in uploaded:
    dst = f"/content/Track4World/input_videos/{fname}"
    with open(dst, "wb") as f:
        f.write(uploaded[fname])
    print(f"已保存: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

VIDEO_PATH = f"/content/Track4World/input_videos/{list(uploaded.keys())[0]}"
print(f"
将使用视频: {VIDEO_PATH}")

In [ ]:
# Cell 5: 推理参数配置 + 执行
# ============ 可修改参数 ============
MODE = "2d"       # "2d" / "3d_ff" / "3d_efep"
IMAGE_SIZE = 448  # T4 16GB 可用 448 或 512
MAX_FRAMES = 50   # T4 可跑更多帧
# ====================================

import os
os.chdir("/content/Track4World")
os.environ["PYTHONIOENCODING"] = "utf-8"

cmd = (
    f"python demo.py"
    f" --mp4_path {VIDEO_PATH}"
    f" --mode {MODE}"
    f" --coordinate camera_base"
    f" --ckpt_init checkpoints/track4world_moge.pth"
    f" --image_size {IMAGE_SIZE}"
    f" --max_frames {MAX_FRAMES}"
)
print(f"执行命令:
{cmd}
")
!{cmd}

In [ ]:
# Cell 6: 打包并下载结果
import os, glob
from google.colab import files

os.chdir("/content/Track4World")
result_dirs = sorted(glob.glob("results/*/"), key=os.path.getmtime, reverse=True)
if not result_dirs:
    print("未找到结果目录，请先运行 Cell 5")
else:
    latest = result_dirs[0].rstrip("/")
    print(f"打包结果目录: {latest}")
    zip_name = f"/content/{latest.replace("/", "_")}_results.zip"
    !zip -r {zip_name} {latest}
    print(f"压缩包大小: {os.path.getsize(zip_name)/1e6:.1f} MB")
    files.download(zip_name)
    print("下载已启动，请查看浏览器下载列表")

## 参数建议（T4 16GB vs 本机 4GB）

| 模式 | image_size | max_frames | T4预计耗时 |
|------|-----------|------------|----------|
| 2d | 448 | 50 | ~3 min |
| 2d | 512 | 30 | ~3 min |
| 3d_ff | 448 | 30 | ~5 min |
| 3d_efep | 448 | 20 | ~6 min |

## Colab 免费版注意事项
- 每天约 ~4小时 GPU 使用时间
- 断开后文件全部丢失，**及时运行 Cell 6 下载结果**
- 第一次安装依赖约需 10~15 分钟，断开重连需重新安装
- 如需更长时间：Colab Pro (0/月) 或 AutoDL 按需计费